In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

%matplotlib inline
from matplotlib import rcParams
rcParams["figure.figsize"] = 11, 8

In [2]:
def fill_nan(table):
    for col in table.columns:
        table[col] = table[col].fillna(table[col].median())
    return table

In [3]:
DATA_PATH = "https://raw.githubusercontent.com/Yorko/mlcourse.ai/main/data/"
data = pd.read_csv(DATA_PATH + "credit_scoring_sample.csv", sep=";")
print(f"Shape: {data.shape}")
data.head()

Shape: (45063, 8)


,SeriousDlqin2yrs,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,NumberOfTimes90DaysLate,NumberOfTime60-89DaysPastDueNotWorse,MonthlyIncome,NumberOfDependents
0,0,64,0,0.249908,0,0,8158.0,0.0
1,0,58,0,3870.000000,0,0,NaN,0.0
2,0,41,0,0.456127,0,0,6666.0,0.0
3,0,43,0,0.000190,0,0,10500.0,2.0
4,1,49,0,0.271820,0,0,400.0,0.0


In [4]:
independent_columns_names = [x for x in data if x != "SeriousDlqin2yrs"]
table = fill_nan(data)
X = table[independent_columns_names]
y = table["SeriousDlqin2yrs"]

## Question 1: There are 5 jurors in a courtroom, each of them can correctly identify the guilt of defendant with 70% probability (independently). What is the probability that jurors will make a correct verdict jointly if the decision is made by majority voting?

In [5]:
from scipy.special import comb

N, p, m = 5, 0.7, 3
probability = sum([comb(N, i, exact=True) * (p**i) * ((1-p)**(N-i)) for i in range(m, N+1)])
print(f"Answer Q1: {probability*100:.2f}%")

Answer Q1: 83.69%


## Question 2: What is the interval estimate of the average age for customers who had payment delay with confidence level of 90%?

In [6]:
def get_bootstrap_samples(data, n_samples):
    indices = np.random.randint(0, len(data), (n_samples, len(data)))
    return data[indices]

def stat_intervals(stat, alpha):
    return np.percentile(stat, [100 * alpha / 2.0, 100 * (1 - alpha / 2.0)])

churn = data[data["SeriousDlqin2yrs"] == 1]["age"].values
np.random.seed(0)
churn_mean_scores = [np.mean(sample) for sample in get_bootstrap_samples(churn, 1000)]
interval = stat_intervals(churn_mean_scores, 0.1)
print(f"Answer Q2: [{interval[0]:.2f}, {interval[1]:.2f}]")

Answer Q2: [45.71, 46.13]


## Logistic Regression Setup

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

lr = LogisticRegression(random_state=5, class_weight="balanced")
parameters = {"C": (0.0001, 0.001, 0.01, 0.1, 1, 10)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)

## Question 3:  Which value of parameter C is optimal?

In [8]:
grid_search = GridSearchCV(lr, parameters, n_jobs=-1, scoring="roc_auc", cv=skf)
grid_search.fit(X, y)

best_c = grid_search.best_params_['C']
print(f"Best C from grid search: {best_c}")

Best C from grid search: 10

Answer Q3: 10


## Question 4: Can we consider the best model stable? See our definition in notebook

In [9]:
# Find index of best C
best_idx = list(parameters['C']).index(best_c)
std_score = grid_search.cv_results_["std_test_score"][best_idx]

is_stable = std_score < 0.005
print(f"Std dev at best C ({best_c}): {std_score:.6f}")
print(f"Is std < 0.005 (0.5%)? {is_stable}")
print(f"Best ROC AUC: {grid_search.best_score_:.6f}")
print(f"\nAnswer Q4: {'Yes' if is_stable else 'No'}")

Std dev at best C (10): 0.007719
Is std < 0.005 (0.5%)? False
Best ROC AUC: 0.795326

Answer Q4: No


## Question 5: What is the most important feature for the best logistic regression model?

In [10]:
from sklearn.preprocessing import StandardScaler

lr_scaled = LogisticRegression(C=best_c, random_state=5, class_weight="balanced")
scaler = StandardScaler()
lr_scaled.fit(scaler.fit_transform(X), y)

feature_importance = pd.DataFrame({
    "feat": independent_columns_names, 
    "coef": lr_scaled.coef_.flatten()
}).sort_values(by="coef", key=abs, ascending=False)

print("Feature importance:")
print(feature_importance)
print(f"\nAnswer Q5: {feature_importance.iloc[0]['feat']}")

Feature importance:
                                   feat      coef
1  NumberOfTime30-59DaysPastDueNotWorse  3.276867
3               NumberOfTimes90DaysLate  3.067259
0                                   age -0.436555
5                         MonthlyIncome -0.203801
6                    NumberOfDependents  0.085393
4  NumberOfTime60-89DaysPastDueNotWorse  0.076824
2                             DebtRatio -0.041773

Answer Q5: NumberOfTime30-59DaysPastDueNotWorse


## Question 6: What is the DebtRatio impact on the prediction?

In [11]:
softmax_values = np.exp(lr_scaled.coef_[0]) / np.sum(np.exp(lr_scaled.coef_[0]))
debt_ratio_idx = independent_columns_names.index('DebtRatio')
debt_ratio_softmax = softmax_values[debt_ratio_idx]

print(f"DebtRatio softmax: {debt_ratio_softmax:.4f}")

DebtRatio softmax: 0.0182


## Question 7: You'll have two estimates for odds of this customer being bad – with original age and with aged increased by 20 years. What's the quotient of this odds? (the second divided by the first).

In [12]:
lr_unscaled = LogisticRegression(C=best_c, random_state=5, class_weight="balanced")
lr_unscaled.fit(X, y)

age_idx = independent_columns_names.index('age')
age_coef = lr_unscaled.coef_[0][age_idx]
odds_ratio = np.exp(age_coef * 20)

print(f"Age coefficient: {age_coef:.6f}")
print(f"Odds ratio for +20 years: {odds_ratio:.4f}")

Age coefficient: -0.015729
Odds ratio for +20 years: 0.7301


## Random Forest

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100, 
    n_jobs=-1, 
    random_state=42, 
    class_weight="balanced"
)

rf_parameters = {
    "max_features": [1, 2, 4],
    "min_samples_leaf": [3, 5, 7, 9],
    "max_depth": [5, 10, 15],
}

## Question 8:  How much higher ROC AUC of the best random forest model than ROC AUC of the best logistic regression on the validation? Select the closest answer.

In [14]:
%%time
rf_grid_search = GridSearchCV(
    rf, rf_parameters, n_jobs=-1, scoring="roc_auc", cv=skf, verbose=True
)
rf_grid_search.fit(X, y)

improvement = rf_grid_search.best_score_ - grid_search.best_score_
print(f"\nLR ROC AUC: {grid_search.best_score_:.6f}")
print(f"RF ROC AUC: {rf_grid_search.best_score_:.6f}")
print(f"Improvement: {improvement:.6f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits

LR ROC AUC: 0.795326
RF ROC AUC: 0.835779
Improvement: 0.040452
CPU times: total: 7.3 s
Wall time: 1min 10s


## Question 9:  What feature has the weakest influence in Random Forest model?

In [15]:
weakest_idx = np.argmin(rf_grid_search.best_estimator_.feature_importances_)
weakest_feature = independent_columns_names[weakest_idx]

rf_importance = pd.DataFrame({
    "feat": independent_columns_names,
    "importance": rf_grid_search.best_estimator_.feature_importances_
}).sort_values(by="importance", ascending=False)

print("RF Feature importance:")
print(rf_importance)

RF Feature importance:
                                   feat  importance
1  NumberOfTime30-59DaysPastDueNotWorse    0.300290
3               NumberOfTimes90DaysLate    0.278749
4  NumberOfTime60-89DaysPastDueNotWorse    0.156534
0                                   age    0.115860
2                             DebtRatio    0.076082
5                         MonthlyIncome    0.057994
6                    NumberOfDependents    0.014491


## Question 11: What the best ROC AUC you've got for bagging with logistic regression as a base estimator?

In [16]:
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import RandomizedSearchCV
import sklearn

# Check sklearn version to use correct parameter name
sklearn_version = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
print(f"sklearn version: {sklearn.__version__}")

if sklearn_version >= (1, 2):
    # New sklearn versions
    print("Using NEW sklearn API (estimator + estimator__C)")
    bg = BaggingClassifier(
        estimator=LogisticRegression(class_weight="balanced"),
        n_estimators=100,
        n_jobs=-1,
        random_state=42,
    )
    bg_parameters = {
        "max_features": [2, 3, 4],
        "max_samples": [0.5, 0.7, 0.9],
        "estimator__C": [0.0001, 0.001, 0.01, 1, 10, 100],
    }
else:
    # Old sklearn versions
    print("Using OLD sklearn API (base_estimator + base_estimator__C)")
    bg = BaggingClassifier(
        base_estimator=LogisticRegression(class_weight="balanced"),
        n_estimators=100,
        n_jobs=-1,
        random_state=42,
    )
    bg_parameters = {
        "max_features": [2, 3, 4],
        "max_samples": [0.5, 0.7, 0.9],
        "base_estimator__C": [0.0001, 0.001, 0.01, 1, 10, 100],
    }

sklearn version: 1.7.2
Using NEW sklearn API (estimator + estimator__C)


In [ ]:
%%time
r_grid_search = RandomizedSearchCV(
    bg,
    bg_parameters,
    n_jobs=-1,
    scoring="roc_auc",
    cv=skf,
    n_iter=20,
    random_state=1,
    verbose=True,
)
r_grid_search.fit(X, y)

print(f"\nBest params: {r_grid_search.best_params_}")
print(f"Best ROC AUC: {r_grid_search.best_score_:.6f}")
print(f"Best ROC AUC %: {r_grid_search.best_score_*100:.2f}%")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
